In [41]:
import scipy as sp
import numpy as np
import pandas as pd
import pickle

pd.set_option('display.max_rows', None)

In [38]:
def mean_confidence_interval(x, confidence=0.95):
    m, se = np.mean(x), sp.stats.sem(x)
    h = se * sp.stats.t.ppf((1 + confidence) / 2., x.shape[0] - 1)
    return np.round(m - h, 2), np.round(m, 2), np.round(m + h, 2)

def preprocess_string(x):
    if x[0] != "-":
        for i, c in enumerate(x):
            if c.isalpha():
                break
    
        x = x[:i] + " " + x[i:]

        if x[0] == "0" and x[1] != ".":
            i = 1
        else:
            for i in range(1, len(x)):
                if (x[i - 1] == "1"):
                    break
        
        return x[:i] + " " + x[i:]
    else:
        return x

In [3]:
with open("./table_results/diff_results_time_uni.pkl", "rb") as f:
    results = pickle.load(f)

mask_condition = results["mask_condition"]
mask_control = results["mask_control"]
mask_both = results["mask_both"]

In [5]:
keys = [x for x in results.keys() if "mask" not in x]
processed_results = {key: {e: [] for e in ["value", "time"]} for key in keys}

for key in keys:
    tmp_mean = np.mean(results[key]["value"][mask_condition], axis=0)
    processed_results[key]["value"] = mean_confidence_interval(tmp_mean)
    
    processed_results[key]["time"] = mean_confidence_interval(np.array(results[key]["time"]))

In [75]:
df = pd.DataFrame(processed_results).transpose()
df.reset_index(inplace=True)
df["index"] = df["index"].apply(preprocess_string)

df["sigma_min"] = df["index"].apply(lambda x: x.split()[0])
df["sigma"] = df["index"].apply(lambda x: x.split()[1])
df["score_type"] = df["index"].apply(lambda x: x.split()[2])

df["value"] = df.value.apply(lambda x: np.round(np.array(x), 2))
df["time"] = df.time.apply(lambda x: np.round(np.array(x), 2))

df.drop("index", inplace=True, axis=1)
df = df[["sigma_min", "sigma", "score_type", "value", "time"]]
df.sort_values(["sigma_min", "sigma", "score_type"], ascending=False, inplace=True)
df

,sigma_min,sigma,score_type,value,time
3,0.1,1,rnl,"[23.96, 24.99, 26.02]","[13.78, 14.53, 15.28]"
2,0.1,1,rl,"[19.53, 20.45, 21.37]","[14.55, 15.35, 16.16]"
1,0.1,1,direct,"[22.64, 23.96, 25.28]","[34.33, 35.09, 35.85]"
18,0.1,0.75,rnl,"[25.46, 26.62, 27.79]","[14.61, 15.35, 16.08]"
17,0.1,0.75,rl,"[21.15, 21.96, 22.77]","[15.04, 15.97, 16.9]"
16,0.1,0.75,direct,"[23.32, 25.17, 27.02]","[37.69, 39.04, 40.38]"
33,0.1,0.5,rnl,"[28.86, 30.02, 31.18]","[15.46, 16.96, 18.47]"
32,0.1,0.5,rl,"[21.45, 21.91, 22.37]","[17.29, 18.09, 18.9]"
31,0.1,0.5,direct,"[25.88, 27.96, 30.04]","[40.9, 42.16, 43.42]"
48,0.1,0.25,rnl,"[31.22, 33.28, 35.35]","[16.58, 17.62, 18.66]"


In [78]:
group = df.groupby(["score_type"])[["value", "time"]].mean().reset_index()
group["value"] = group.value.apply(lambda x: np.round(np.array(x), 2))
group["time"] = group.time.apply(lambda x: np.round(np.array(x), 2))
group

,score_type,value,time
0,direct,"[23.41, 25.72, 28.03]","[41.59, 43.02, 44.45]"
1,rl,"[20.71, 21.75, 22.8]","[18.12, 19.28, 20.45]"
2,rnl,"[34.15, 37.09, 40.03]","[15.29, 16.39, 17.48]"
3,true,"[20.16, 20.16, 20.16]","[0.01, 0.01, 0.02]"


In [97]:
group = df[df.score_type.isin(["rl", "direct"])].copy()
group["value"] = group.value.apply(lambda x: x - np.array([20.16, 20.16, 20.16]))
group["value"] = group.value.apply(lambda x: np.round(np.array(x), 2))
group["time"] = group.time.apply(lambda x: np.round(np.array(x), 2))
group

,sigma_min,sigma,score_type,value,time
2,0.1,1,rl,"[-0.63, 0.29, 1.21]","[14.55, 15.35, 16.16]"
1,0.1,1,direct,"[2.48, 3.8, 5.12]","[34.33, 35.09, 35.85]"
17,0.1,0.75,rl,"[0.99, 1.8, 2.61]","[15.04, 15.97, 16.9]"
16,0.1,0.75,direct,"[3.16, 5.01, 6.86]","[37.69, 39.04, 40.38]"
32,0.1,0.5,rl,"[1.29, 1.75, 2.21]","[17.29, 18.09, 18.9]"
31,0.1,0.5,direct,"[5.72, 7.8, 9.88]","[40.9, 42.16, 43.42]"
47,0.1,0.25,rl,"[1.71, 2.74, 3.76]","[19.09, 19.96, 20.83]"
46,0.1,0.25,direct,"[5.55, 8.87, 12.18]","[42.84, 44.08, 45.31]"
62,0.1,0,rl,"[2.11, 3.44, 4.78]","[20.14, 21.22, 22.3]"
61,0.1,0,direct,"[9.08, 11.62, 14.17]","[45.91, 46.48, 47.05]"


In [79]:
df[df.score_type == "rl"]

,sigma_min,sigma,score_type,value,time
2,0.1,1,rl,"[19.53, 20.45, 21.37]","[14.55, 15.35, 16.16]"
17,0.1,0.75,rl,"[21.15, 21.96, 22.77]","[15.04, 15.97, 16.9]"
32,0.1,0.5,rl,"[21.45, 21.91, 22.37]","[17.29, 18.09, 18.9]"
47,0.1,0.25,rl,"[21.87, 22.9, 23.92]","[19.09, 19.96, 20.83]"
62,0.1,0,rl,"[22.27, 23.6, 24.94]","[20.14, 21.22, 22.3]"
5,0.01,1,rl,"[18.73, 19.45, 20.16]","[13.87, 14.66, 15.46]"
20,0.01,0.75,rl,"[19.75, 20.68, 21.61]","[16.9, 18.42, 19.95]"
35,0.01,0.5,rl,"[21.51, 22.14, 22.77]","[19.87, 20.69, 21.5]"
50,0.01,0.25,rl,"[22.9, 24.24, 25.59]","[19.43, 20.23, 21.03]"
65,0.01,0,rl,"[24.81, 26.08, 27.36]","[20.6, 21.65, 22.7]"


In [100]:
group = df.copy()
group["value"] = group.value.apply(lambda x: x - np.array([20.16, 20.16, 20.16]))
group = group.groupby(["sigma_min", "score_type"])[["value", "time"]].mean().reset_index()
group["value"] = group.value.apply(lambda x: np.round(np.array(x), 2))
group["time"] = group.time.apply(lambda x: np.round(np.array(x), 2))
group.sort_values(["sigma_min", "score_type"], ascending=False, inplace=True)
group

,sigma_min,score_type,value,time
15,0.1,rnl,"[8.89, 10.57, 12.26]","[15.28, 16.28, 17.29]"
14,0.1,rl,"[1.09, 2.0, 2.91]","[17.22, 18.12, 19.02]"
13,0.1,direct,"[5.2, 7.42, 9.64]","[40.33, 41.37, 42.4]"
12,0.01,rnl,"[13.9, 16.74, 19.59]","[15.15, 16.1, 17.05]"
11,0.01,rl,"[1.38, 2.36, 3.34]","[18.13, 19.13, 20.13]"
10,0.01,direct,"[4.65, 7.77, 10.88]","[42.27, 43.26, 44.24]"
9,0.001,rnl,"[16.57, 19.24, 21.9]","[15.16, 16.32, 17.49]"
8,0.001,rl,"[0.48, 1.46, 2.43]","[18.01, 19.13, 20.25]"
7,0.001,direct,"[3.54, 5.52, 7.5]","[42.12, 43.21, 44.3]"
6,0.0001,rnl,"[16.93, 20.5, 24.07]","[15.53, 16.71, 17.89]"


In [101]:
group = df.copy()
group["value"] = group.value.apply(lambda x: x - np.array([20.16, 20.16, 20.16]))
group = group.groupby(["sigma", "score_type"])[["value", "time"]].mean().reset_index()
group["value"] = group.value.apply(lambda x: np.round(np.array(x), 2))
group["time"] = group.time.apply(lambda x: np.round(np.array(x), 2))
group.sort_values(["sigma", "score_type"], ascending=False, inplace=True)
group

,sigma,score_type,value,time
15,1,rnl,"[4.31, 5.67, 7.02]","[13.62, 14.63, 15.65]"
14,1,rl,"[-2.03, -1.11, -0.19]","[14.6, 15.69, 16.78]"
13,1,direct,"[0.86, 2.46, 4.05]","[35.74, 36.88, 38.02]"
12,0.75,rnl,"[5.31, 7.01, 8.7]","[15.61, 16.39, 17.17]"
11,0.75,rl,"[-0.85, 0.09, 1.04]","[16.81, 18.06, 19.31]"
10,0.75,direct,"[1.01, 2.59, 4.17]","[39.72, 40.97, 42.21]"
9,0.5,rnl,"[8.13, 10.23, 12.32]","[16.6, 18.0, 19.41]"
8,0.5,rl,"[0.45, 1.48, 2.51]","[18.92, 20.26, 21.6]"
7,0.5,direct,"[2.72, 4.83, 6.93]","[41.83, 44.47, 47.1]"
6,0.25,rnl,"[12.55, 15.39, 18.23]","[16.58, 17.68, 18.79]"


In [104]:
df["diff_mean"] = df.value.apply(lambda x: x[1] - 20.16)
df["diff_mean_abs"] = df.diff_mean.apply(abs)
df.sort_values("diff_mean_abs")

,sigma_min,sigma,score_type,value,time,diff_mean,diff_mean_abs
0,-,-,true,"[20.16, 20.16, 20.16]","[0.01, 0.01, 0.02]",0.00,0.00
2,0.1,1,rl,"[19.53, 20.45, 21.37]","[14.55, 15.35, 16.16]",0.29,0.29
23,0.001,0.75,rl,"[18.9, 19.71, 20.52]","[16.95, 18.41, 19.87]",-0.45,0.45
29,0,0.75,rl,"[18.31, 19.66, 21.02]","[17.94, 18.76, 19.57]",-0.50,0.50
20,0.01,0.75,rl,"[19.75, 20.68, 21.61]","[16.9, 18.42, 19.95]",0.52,0.52
5,0.01,1,rl,"[18.73, 19.45, 20.16]","[13.87, 14.66, 15.46]",-0.71,0.71
10,0.0001,1,direct,"[20.1, 21.04, 21.98]","[35.12, 36.99, 38.87]",0.88,0.88
26,0.0001,0.75,rl,"[18.43, 19.24, 20.06]","[17.21, 18.74, 20.27]",-0.92,0.92
28,0,0.75,direct,"[18.75, 21.11, 23.46]","[40.29, 41.65, 43.0]",0.95,0.95
41,0.0001,0.5,rl,"[19.44, 21.27, 23.1]","[18.34, 20.39, 22.45]",1.11,1.11
